# Community Structure Invisible to SQL Pairs

This notebook demonstrates the second structural gap — Genie's inability to resolve community boundaries using SQL.

The check asks Genie to find clusters of accounts transferring heavily among themselves — the same question a fraud analyst would ask when investigating coordinated activity. Genie returns bilateral pairs (accounts that exchange money with each other), while the actual fraud structure is a 100-account ring with dense internal transfers that Louvain resolves as a single community.

**Four phases:**

- **Phase 1** — Call Genie and display the result
- **Phase 2** — Load the ground truth ring membership from the UC Volume
- **Phase 3** — Compute the max ring coverage metric and determine whether the structural gap is confirmed
- **Phase 4** — Explain why SQL bilateral pairs cannot replicate Louvain community detection

**Before running:** Set your `SPACE_ID` in the configuration cell below. Run `setup/upload_and_create_tables.sh` first to load the tables and upload `ground_truth.json` to the UC Volume.

**Confirmation threshold:** `max_ring_coverage < 0.50` — the gap is confirmed when Genie cannot form a group covering more than half of any fraud ring. When Genie returns only pairs (no grouping column), `max_ring_coverage = 0.0` and the gap is confirmed, showing the community boundary is invisible to SQL.

In [ ]:
%pip install --upgrade databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
SPACE_ID = dbutils.secrets.get("neo4j-graph-engineering", "genie_space_id")

CATALOG = "graph-enriched-lakehouse"
SCHEMA  = "graph-enriched-schema"
VOLUME  = "graph-enriched-volume"

In [ ]:
from databricks.sdk import WorkspaceClient
import pandas as pd

from demo_utils import genie_caller, load_ground_truth, build_ring_lookup, check_community_structure

w = WorkspaceClient()
print("Connected to:", w.config.host)

ask_genie = genie_caller(w, SPACE_ID)

## Phase 1 — Call Genie and Display Results

The question below asks Genie to find clusters of accounts that predominantly transfer money among themselves. This is the community detection question — the analyst is looking for coordinated activity, not individual suspicious accounts.

Genie's best SQL approximation is a bidirectional pair query: find pairs of accounts that send money to each other, ranked by how frequently they exchange transfers. The result is a list of bilateral relationships — account A and account B with 3–4 mutual transfers. If Genie attempts to group these pairs into clusters, it produces small collections of a few accounts, not the 100-account rings that Louvain resolves.

The result looks like it found something. The account pairs it surfaces are often real ring members — they do exchange money with each other. What the result cannot show is that those two accounts belong to a 100-member community with a distinct boundary separating it from adjacent rings.

In [ ]:
CHECK_2_QUESTION = (
    "Which groups of accounts are transferring money heavily among themselves? "
    "Try to identify clusters of accounts that predominantly send money to each other."
)

print("Question:", CHECK_2_QUESTION)
print()

response = ask_genie(CHECK_2_QUESTION)
print(f"Status:  {response['status']}")

if response["text"] and response["df"] is None:
    print(f"\nGenie returned a text response instead of a table:\n{response['text']}")
    raise RuntimeError(
        "Genie did not return a query result. "
        "Confirm the Genie Space is connected to the correct tables and try again."
    )

print(f"Rows returned: {len(response['df'])}")

In [ ]:
# Genie's answer — the raw result before any comparison
print("Genie returned these account pairs or clusters:\n")
display(response["df"])

In [ ]:
# The SQL Genie generated — shows what Genie actually measured
print("SQL Genie generated:\n")
print(response["sql"])

## Phase 2 — Load Ground Truth Ring Membership

The ten fraud rings each contain 100 accounts with a dense internal transfer network. `ground_truth.json` records the full ring membership — uploaded to the UC Volume by `setup/upload_and_create_tables.sh`.

Loading this file gives us the ring membership needed to evaluate whether any group Genie identified covers a meaningful fraction of a real ring.

In [ ]:
GROUND_TRUTH_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/ground_truth.json"

gt = load_ground_truth(GROUND_TRUTH_PATH)
_, whale_ids = build_ring_lookup(gt)

rings             = gt["rings"]
total_rings       = len(rings)
accounts_per_ring = len(rings[0]["account_ids"]) if rings else 0
total_fraud       = sum(len(r["account_ids"]) for r in rings)

print(f"Ground truth loaded from: {GROUND_TRUTH_PATH}")
print(f"  Rings:                {total_rings}")
print(f"  Accounts per ring:    {accounts_per_ring}")
print(f"  Total fraud accounts: {total_fraud:,}")
print(f"  Whale accounts:       {len(whale_ids):,}")
print()
print(f"Pass criterion: max_ring_coverage < 50%")
print(f"  The check passes when Genie cannot form a group covering more than")
print(f"  half of any fraud ring. Each ring has {accounts_per_ring} members.")

## Phase 3 — Compute the Max Ring Coverage Metric

The metric for this check is `max_ring_coverage`: across all groups or clusters Genie returned, for each identified group, count what fraction of any fraud ring it covers. Take the maximum across all groups and all rings.

- **Genie returns only pairs (no grouping column):** `max_ring_coverage = 0.0`. Genie identified no communities at all. **This confirms the structural gap.** SQL bilateral pairs carry no concept of a community boundary.
- **Genie returns groups with low coverage (< 50%):** Genie attempted to form clusters but each covers only a small fraction of any ring. The community boundary is still not visible. **This also confirms the structural gap.**
- **Genie returns groups with high coverage (≥ 50%):** Genie covered more than half of a fraud ring in a single identified group — possibly via a transitive-closure CTE. The community-vs-pairs gap is narrower than intended.

Confirmation threshold: `max_ring_coverage < 0.50`

The key distinction from a simple account-exposure metric: this measures whether Genie formed *groups*, not just whether ring accounts happened to appear in high-volume pairs. Finding ring accounts in pairs is expected — ring members do exchange money. What Genie cannot do is label those accounts as belonging to a community with a shared boundary.

In [ ]:
df = response["df"]

# Detect whether Genie returned a grouping structure or only pairs.
group_col = next(
    (c for c in df.columns if any(kw in c.lower() for kw in ("cluster", "community", "group"))),
    None,
)
account_cols = [c for c in df.columns if "account" in c.lower()]

if group_col and account_cols:
    print("Genie returned a grouping structure.")
    print(f"  Group column:    '{group_col}'")
    print(f"  Account column:  '{account_cols[0]}'")
    print(f"  Distinct groups: {df[group_col].nunique()}")
else:
    print("Genie returned pairs — no community grouping column detected.")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Rows:    {len(df)}")

In [ ]:
ring_lists = [r["account_ids"] for r in gt["rings"]]
result = check_community_structure(response["df"], ring_lists)

print("Community structure analysis:")
print(f"  Structure type:    {result['structure_type']}")
print(f"  Groups returned:   {result['groups_returned']}")
print(f"  Total rows:        {result['total_rows']}")
print()
print(f"  Max ring coverage: {result['max_ring_coverage']:.1%}")
print(f"  Confirmation threshold: < 50%")

## Phase 4 — Why SQL Bilateral Pairs Cannot Replicate Louvain

The gap confirmed by this check is not a limitation of Genie — it is a limitation of SQL itself for this class of problem.

### What Genie can do

Genie can write a bidirectional pair query: find accounts where A transfers to B and B transfers to A, sort by transfer count. It can also attempt transitive closure — a recursive CTE that chains pairs together. Both of these find ring members exchanging money. Neither can compute community boundaries.

### What Louvain does that SQL cannot

Louvain runs iterative global modularity optimization. In each iteration, every node evaluates whether moving to a neighboring community would increase the global modularity score — a measure of how much denser the within-community edges are compared to a statistical null model. The algorithm updates all assignments simultaneously and repeats until global modularity converges.

Two structural consequences follow:

1. **SQL transitive closure merges rings that share any cross-ring edge.** If ring 3 and ring 7 happen to share one background transfer, a recursive CTE chains them into a single component. Louvain's modularity objective evaluates the entire graph state at once: even with a shared edge, if the internal densities of rings 3 and 7 are high and distinct, Louvain keeps them separated because merging them would reduce global modularity.

2. **The boundary computation requires the full graph.** Louvain draws community boundaries based on every node's relationship to every other node in the graph. SQL can only evaluate local relationships — the rows it has been given. There is no SQL formulation that globally optimizes across all accounts simultaneously without materializing the full graph.

### What an analyst sees

The bilateral pairs in Phase 1 look meaningful — they are real ring members exchanging money. What the analyst cannot tell from the pairs alone is that those two accounts belong to a 100-member group with a shared boundary distinct from neighboring rings. The pairs provide no basis for deciding where the community ends. Louvain's `community_id` column provides exactly that.

In [ ]:
print("=" * 62)
print("COMMUNITY STRUCTURE SUMMARY")
print("=" * 62)
print()
print(f"Genie returned {result['total_rows']} rows.")
print(f"Structure type: {result['structure_type']}")
print()

if result["structure_type"] == "pairs":
    print("Genie returned only bilateral pairs — no grouping column.")
    print("max_ring_coverage = 0.0  (no communities identified)")
else:
    print(f"Genie identified {result['groups_returned']} distinct groups.")
    print(f"Max ring coverage: {result['max_ring_coverage']:.1%}")
    print(f"  The best group Genie formed covers {result['max_ring_coverage']:.1%}")
    print(f"  of a {accounts_per_ring}-account fraud ring.")

print()
print("-" * 62)

if result["passed"]:
    print("STRUCTURAL GAP CONFIRMED")
    print("-" * 62)
    print()
    if result["structure_type"] == "pairs":
        print("Genie returned pairs with no community grouping.")
        print("The 100-account community boundary is invisible to SQL.")
        print("This confirms the gap that Louvain closes after GDS enrichment.")
    else:
        print(f"Genie's groups cover at most {result['max_ring_coverage']:.1%} of any fraud ring.")
        print("No group comes close to the 100-account community boundary.")
        print("This confirms the gap that Louvain closes after GDS enrichment.")
else:
    print("STRUCTURAL GAP NOT CONFIRMED")
    print("-" * 62)
    print()
    print(f"max_ring_coverage {result['max_ring_coverage']:.1%} >= 50%.")
    print("Genie formed a group covering more than half a fraud ring.")
    print("Genie may have written a transitive-closure CTE that chains")
    print("pairs into communities. The Louvain gap is narrower than intended.")
    print("Re-run setup/verify_fraud_patterns.py to check structural properties.")